In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_2.py ──
"""
Shared infrastructure for MLFP02 Exercise 2 — Parameter Estimation and Inference.

Contains: Singapore economic data loading, log-likelihood objectives for
Normal / Student-t / Laplace, profile LR CI helpers, bootstrap utilities,
plot output directory and save helper.

Technique-specific narrative (which scenario, which interpretation) belongs
in the per-technique files in ``modules/mlfp02/solutions/ex_2/``.
"""

from pathlib import Path
from typing import Callable

import numpy as np
import plotly.graph_objects as go
import polars as pl
from scipy import stats
from scipy.optimize import minimize


# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp02_ex2_mle_theory"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig: go.Figure, filename: str) -> Path:
    """Write a Plotly figure to the exercise output directory."""
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore economic indicators
# ════════════════════════════════════════════════════════════════════════

SINGAPORE_ECON_DATASET = ("mlfp01", "economic_indicators.csv")


def load_singapore_econ() -> pl.DataFrame:
    """Load Singapore economic indicators (GDP, inflation, unemployment)."""
    loader = MLFPDataLoader()
    return loader.load(*SINGAPORE_ECON_DATASET)


def extract_series(df: pl.DataFrame, column: str) -> np.ndarray:
    """Drop nulls and return a float64 numpy array for a given column."""
    return df[column].drop_nulls().to_numpy().astype(np.float64)


def load_gdp_growth() -> np.ndarray:
    """Convenience: GDP growth (annualised quarterly %) as a 1-D numpy array."""
    return extract_series(load_singapore_econ(), "gdp_growth_pct")


# ════════════════════════════════════════════════════════════════════════
# LIKELIHOOD OBJECTIVES
# ════════════════════════════════════════════════════════════════════════
#
# All objectives are NEGATIVE log-likelihoods so they can be passed
# directly to scipy.optimize.minimize. Location/scale parameters are
# reparameterised where needed (log-sigma) to keep the optimiser in a
# valid region without explicit bounds.


def neg_log_likelihood_normal(params: np.ndarray, x: np.ndarray) -> float:
    """-log L for X ~ N(mu, sigma^2). params = [mu, log_sigma]."""
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    if sigma <= 0:
        return np.inf
    return float(-np.sum(stats.norm.logpdf(x, loc=mu, scale=sigma)))


def neg_log_likelihood_t(params: np.ndarray, x: np.ndarray) -> float:
    """-log L for X ~ t(df, mu, scale). params = [df, mu, scale]."""
    df, mu, scale = params
    if df <= 0 or scale <= 0:
        return np.inf
    return float(-np.sum(stats.t.logpdf(x, df=df, loc=mu, scale=scale)))


def fit_normal_mle(x: np.ndarray) -> dict:
    """Fit Normal MLE numerically via L-BFGS-B. Returns mu, sigma, loglik, result."""
    x0 = np.array([float(x.mean()), float(np.log(x.std() + 1e-8))])
    result = minimize(
        neg_log_likelihood_normal,
        x0,
        args=(x,),
        method="L-BFGS-B",
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    return {
        "mu": float(result.x[0]),
        "sigma": float(np.exp(result.x[1])),
        "loglik": float(-result.fun),
        "converged": bool(result.success),
        "result": result,
    }


def fit_student_t_mle(x: np.ndarray) -> dict:
    """Fit Student-t MLE via Nelder-Mead. Returns df, mu, scale, loglik."""
    x0 = np.array([5.0, float(x.mean()), float(x.std() + 1e-8)])
    result = minimize(neg_log_likelihood_t, x0, args=(x,), method="Nelder-Mead")
    return {
        "df": float(result.x[0]),
        "mu": float(result.x[1]),
        "scale": float(result.x[2]),
        "loglik": float(-result.fun),
        "converged": bool(result.success),
        "result": result,
    }


# ════════════════════════════════════════════════════════════════════════
# FISHER INFORMATION + CONFIDENCE INTERVALS
# ════════════════════════════════════════════════════════════════════════
#
# For N(mu, sigma^2):
#   Var(mu_hat)   = sigma^2 / n         =>  SE(mu_hat)  = sigma / sqrt(n)
#   Var(sigma_hat) ~ sigma^2 / (2n)     =>  SE(sigma_hat) = sigma / sqrt(2n)
#
# These come directly from inverting the Fisher information matrix at
# the MLE.


def normal_fisher_standard_errors(sigma: float, n: int) -> tuple[float, float]:
    """Return (SE_mu, SE_sigma) from Fisher information at the Normal MLE."""
    se_mu = sigma / np.sqrt(n)
    se_sigma = sigma / np.sqrt(2 * n)
    return float(se_mu), float(se_sigma)


def wald_ci(estimate: float, se: float, alpha: float = 0.05) -> tuple[float, float]:
    """Symmetric Wald CI: estimate ± z_{1-alpha/2} * SE."""
    z = float(stats.norm.ppf(1 - alpha / 2))
    return (estimate - z * se, estimate + z * se)


def profile_lr_ci_normal_mu(
    x: np.ndarray,
    mu_hat: float,
    sigma_hat: float,
    loglik_at_mle: float,
    alpha: float = 0.05,
    grid_width_in_se: float = 4.0,
    n_grid: int = 500,
) -> tuple[tuple[float, float], np.ndarray, np.ndarray]:
    """Profile likelihood 1-alpha CI for the Normal mean.

    sigma is a NUISANCE parameter, so it is profiled out: at every grid
    value of mu we re-maximise the likelihood over sigma, which for the
    Normal has the closed form sigma_hat(mu)^2 = mean((x - mu)^2). The
    profile log-likelihood is l_p(mu) = l(mu, sigma_hat(mu)).

    (Holding sigma fixed at the global MLE instead would give
    2*(l_max - l(mu)) = n*(x_bar - mu)^2 / sigma_hat^2 — exactly the Wald
    statistic, i.e. no profile at all.)

    The CI is the set of mu where 2*(loglik_at_mle - l_p(mu)) <= chi^2_{1-alpha, df=1}.

    Returns (ci, mu_grid, profile_loglik_values) so the caller can plot it.
    """
    n = len(x)
    se_mu = sigma_hat / np.sqrt(n)
    threshold = float(stats.chi2.ppf(1 - alpha, df=1)) / 2.0

    mu_grid = np.linspace(
        mu_hat - grid_width_in_se * se_mu,
        mu_hat + grid_width_in_se * se_mu,
        n_grid,
    )
    loglik_values = np.array(
        [
            -neg_log_likelihood_normal(
                [mu, 0.5 * np.log(np.mean((x - mu) ** 2))], x
            )
            for mu in mu_grid
        ]
    )
    lr_values = loglik_at_mle - loglik_values
    mask = lr_values <= threshold
    if not mask.any():
        raise ValueError(
            "profile likelihood: no grid point inside the CI — check that "
            "loglik_at_mle is the log-likelihood at (mu_hat, sigma_hat)"
        )
    if mask[0] or mask[-1]:
        raise ValueError(
            "profile likelihood: CI reaches the grid edge — increase "
            "grid_width_in_se"
        )
    ci = (float(mu_grid[mask][0]), float(mu_grid[mask][-1]))
    return ci, mu_grid, loglik_values


# ════════════════════════════════════════════════════════════════════════
# MAP — NORMAL LIKELIHOOD + NORMAL PRIOR ON THE MEAN
# ════════════════════════════════════════════════════════════════════════


def make_map_objective(
    prior_mean: float, prior_std: float
) -> Callable[[np.ndarray, np.ndarray], float]:
    """Return a neg_map_objective(params, x) closure for the given Normal prior on mu."""

    def neg_map_objective(params: np.ndarray, x: np.ndarray) -> float:
        mu, log_sigma = params
        sigma = np.exp(log_sigma)
        if sigma <= 0:
            return np.inf
        nll = -np.sum(stats.norm.logpdf(x, loc=mu, scale=sigma))
        neg_log_prior = -stats.norm.logpdf(mu, loc=prior_mean, scale=prior_std)
        return float(nll + neg_log_prior)

    return neg_map_objective


def fit_normal_map(x: np.ndarray, prior_mean: float, prior_std: float) -> dict:
    """Fit MAP for Normal likelihood with a Normal(prior_mean, prior_std^2) prior on mu."""
    neg_map = make_map_objective(prior_mean, prior_std)
    x0 = np.array([float(x.mean()), float(np.log(x.std() + 1e-8))])
    result = minimize(
        neg_map,
        x0,
        args=(x,),
        method="L-BFGS-B",
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    return {
        "mu": float(result.x[0]),
        "sigma": float(np.exp(result.x[1])),
        "converged": bool(result.success),
        "result": result,
    }


# ════════════════════════════════════════════════════════════════════════
# BOOTSTRAP UTILITIES
# ════════════════════════════════════════════════════════════════════════


def bootstrap_statistic(
    x: np.ndarray,
    statistic: Callable[[np.ndarray], float],
    n_boot: int = 10_000,
    seed: int | None = 42,
) -> np.ndarray:
    """Nonparametric bootstrap: resample x with replacement, apply statistic."""
    rng = np.random.default_rng(seed)
    n = len(x)
    out = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        out[i] = statistic(rng.choice(x, size=n, replace=True))
    return out


def bootstrap_percentile_ci(
    boot_samples: np.ndarray, alpha: float = 0.05
) -> tuple[float, float]:
    lo = float(np.percentile(boot_samples, 100 * alpha / 2))
    hi = float(np.percentile(boot_samples, 100 * (1 - alpha / 2)))
    return lo, hi


# ════════════════════════════════════════════════════════════════════════
# AIC / BIC
# ════════════════════════════════════════════════════════════════════════


def aic(k: int, loglik: float) -> float:
    return 2 * k - 2 * loglik


def bic(k: int, loglik: float, n: int) -> float:
    return k * float(np.log(n)) - 2 * loglik


# ════════════════════════════════════════════════════════════════════════
# TAXI TRIPS — count and duration data for Poisson / Exponential models
# ════════════════════════════════════════════════════════════════════════
#
# The GDP series suits location-scale families (Normal, Student-t, Laplace).
# Counts (trips per hour) and positive durations (seconds between pickups)
# need the Poisson-process pair: Poisson for counts, Exponential (or Gamma,
# Weibull) for waiting times. Data: Singapore taxi trips (mlfp01).

TAXI_DATASET = ("mlfp01", "sg_taxi_trips.parquet")
TAXI_WINDOW_YEAR: int = 2024
TAXI_WINDOW_MONTH: int = 4  # April 2024 — a complete, busy month


def load_taxi_trips() -> pl.DataFrame:
    """Load Singapore taxi trips with parsed pickup timestamps."""
    loader = MLFPDataLoader()
    trips = loader.load(*TAXI_DATASET)
    return trips.with_columns(
        pl.col("pickup_datetime").str.to_datetime(strict=False).alias("pickup_ts")
    ).drop_nulls("pickup_ts")


def taxi_trips_per_hour(
    trips: pl.DataFrame, year: int = TAXI_WINDOW_YEAR, month: int = TAXI_WINDOW_MONTH
) -> np.ndarray:
    """Trip counts per (date, hour) cell within one calendar month."""
    window = trips.filter(
        (pl.col("pickup_ts").dt.year() == year)
        & (pl.col("pickup_ts").dt.month() == month)
    )
    counts = window.group_by(
        pl.col("pickup_ts").dt.date().alias("date"),
        pl.col("pickup_ts").dt.hour().alias("hour"),
    ).len()
    return counts["len"].to_numpy().astype(np.float64)


def taxi_interarrival_seconds(
    trips: pl.DataFrame, year: int = TAXI_WINDOW_YEAR, month: int = TAXI_WINDOW_MONTH
) -> np.ndarray:
    """Seconds between consecutive pickups within one calendar month."""
    window = trips.filter(
        (pl.col("pickup_ts").dt.year() == year)
        & (pl.col("pickup_ts").dt.month() == month)
    ).sort("pickup_ts")
    ts = window["pickup_ts"].to_numpy()
    gaps = np.diff(ts).astype("timedelta64[s]").astype(np.float64)
    return gaps[gaps > 0]


def taxi_fares(trips: pl.DataFrame) -> np.ndarray:
    """Fare amounts (SGD) as a float64 array — the LLN demonstration series."""
    return trips["fare_sgd"].drop_nulls().to_numpy().astype(np.float64)


# ════════════════════════════════════════════════════════════════════════
# POISSON / EXPONENTIAL / GAMMA / WEIBULL MLE
# ════════════════════════════════════════════════════════════════════════


def poisson_mle(counts: np.ndarray) -> dict:
    """Closed-form Poisson MLE: lambda_hat = sample mean; loglik at the MLE."""
    lam = float(np.mean(counts))
    loglik = float(np.sum(stats.poisson.logpmf(counts.astype(int), lam)))
    return {"lambda": lam, "loglik": loglik, "n": len(counts), "k": 1}


def exponential_mle(durations: np.ndarray) -> dict:
    """Closed-form Exponential MLE: lambda_hat = 1 / sample mean."""
    lam = 1.0 / float(np.mean(durations))
    loglik = float(np.sum(stats.expon.logpdf(durations, scale=1.0 / lam)))
    return {"lambda": lam, "loglik": loglik, "n": len(durations), "k": 1}


def gamma_mle(durations: np.ndarray) -> dict:
    """Gamma MLE via scipy with loc pinned at 0 (durations are positive)."""
    shape, _, scale = stats.gamma.fit(durations, floc=0)
    loglik = float(np.sum(stats.gamma.logpdf(durations, shape, loc=0, scale=scale)))
    return {
        "shape": float(shape),
        "scale": float(scale),
        "loglik": loglik,
        "n": len(durations),
        "k": 2,
    }


def weibull_mle(durations: np.ndarray) -> dict:
    """Weibull MLE via scipy with loc pinned at 0."""
    shape, _, scale = stats.weibull_min.fit(durations, floc=0)
    loglik = float(
        np.sum(stats.weibull_min.logpdf(durations, shape, loc=0, scale=scale))
    )
    return {
        "shape": float(shape),
        "scale": float(scale),
        "loglik": loglik,
        "n": len(durations),
        "k": 2,
    }


# ════════════════════════════════════════════════════════════════════════
# EXPERIMENT TRACKING — kailash-ml ExperimentTracker
# ════════════════════════════════════════════════════════════════════════
#
# Every technique logs its fitted model the moment it is trained — params
# first, metrics second — so the run is auditable even if the session dies
# before the report is written. DataFlow rewrites relative sqlite URLs, so
# the store URL is pinned to an absolute path beside the exercise outputs.

TRACKER_STORE_URL = (
    f"sqlite:///{(OUTPUT_DIR / 'experiments.db').resolve().as_posix()}"
)


def track_train_run(
    experiment: str,
    run_name: str,
    params: dict[str, str],
    metrics: dict[str, float],
) -> str:
    """Log one Train-phase run to ExperimentTracker (sync wrapper).

    Returns the run_id. The tracker is closed in a finally block — kailash-ml
    holds the store connection open until close() is called.
    """
    import asyncio

    async def _log() -> str:
        from kailash_ml import ExperimentTracker

        tracker = await ExperimentTracker.create(store_url=TRACKER_STORE_URL)
        try:
            async with tracker.track(
                experiment=experiment, run_name=run_name
            ) as run:
                await run.log_params(params)
                await run.log_metrics(metrics)
                return run.run_id
        finally:
            await tracker.close()

    return asyncio.run(_log())


# ════════════════════════════════════════════════════════════════════════
# DEFAULTS — SAMPLE SIZES, SEEDS, PRIOR VALUES
# ════════════════════════════════════════════════════════════════════════
#
# These constants are referenced by multiple technique files. Change them
# once here and every file picks up the update.

DEFAULT_SEED: int = 42
DEFAULT_N_BOOT: int = 10_000
DEFAULT_N_CLT_REPS: int = 5000

# Singapore prior on quarterly GDP growth — long-run open-economy estimate
# that we use to illustrate MAP shrinkage.
GDP_PRIOR_MEAN: float = 3.5  # %
GDP_PRIOR_STD: float = 1.5  # %


# ── shared/mlfp02/ex_3.py ──
"""
Shared infrastructure for MLFP02 Exercise 3 — A/B testing & multiple comparisons.

Contains: experiment data loading, group extraction, SRM check, common constants,
and small statistical helpers reused across the four technique files:

    01_bootstrap_power.py    — bootstrap CIs + MDE + power curves
    02_hypothesis_testing.py — two-proportion z-test + effect sizes
    03_multiple_testing.py   — Bonferroni + BH-FDR + FDR simulation
    04_permutation_test.py   — distribution-free alternative
    05_parametric_bootstrap.py — resample from a fitted model vs the data
    06_one_sample_one_tailed.py — one-sample t + directional tests

Technique-specific code (the actual corrections, permutation loops, power
formulas) does NOT belong here — each technique file owns its own logic.
"""

from pathlib import Path
from typing import Any, Callable

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ════════════════════════════════════════════════════════════════════════

ALPHA: float = 0.05
POWER_TARGET: float = 0.80
N_BOOTSTRAP: int = 10_000
N_PERMUTATIONS: int = 10_000
RANDOM_SEED: int = 42

OUTPUT_DIR = Path("outputs") / "mlfp02_ex3_ab_testing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The experiment was DESIGNED with unequal allocation across four arms.
# SRM must be tested against this design, never against a default 50/50.
DESIGNED_ALLOCATION: dict[str, float] = {
    "control": 0.40,
    "treatment_a": 0.35,
    "treatment_b": 0.15,
    "variant_c": 0.10,
}

# Exercise 3 compares ONE treatment arm with control. Pooling several
# different treatments into one "treatment" group would estimate a
# meaningless mixture of effects.
TREATMENT_ARM: str = "treatment_a"

# Binary success event: a "qualifying order" — a basket of at least $50
# (metric_value is basket value in SGD). Nearly every user has
# metric_value > 0, so "> 0" would give a ~99% "conversion" rate.
CONVERSION_THRESHOLD: float = 50.0


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore e-commerce A/B test
# ════════════════════════════════════════════════════════════════════════


def load_experiment_all() -> pl.DataFrame:
    """Load every arm of the experiment (control + three variants).

    Columns: user_id, experiment_group, metric_value, pre_metric_value,
             revenue, timestamp, segment, platform, country, converted.

    `converted` = 1 when the user placed a qualifying order
    (metric_value >= CONVERSION_THRESHOLD).
    """
    loader = MLFPDataLoader()
    df = loader.load("mlfp02", "experiment_data.parquet")
    return df.with_columns(
        (pl.col("metric_value") >= CONVERSION_THRESHOLD)
        .cast(pl.Int8)
        .alias("converted")
    )


def designed_control_share(arm: str = TREATMENT_ARM) -> float:
    """Designed share of control within the (control, arm) pair."""
    c = DESIGNED_ALLOCATION["control"]
    return c / (c + DESIGNED_ALLOCATION[arm])


def load_experiment(arm: str = TREATMENT_ARM) -> pl.DataFrame:
    """Control vs ONE treatment arm, with a `group` column in {control, treatment}.

    Only these two arms are kept, so every downstream comparison is a
    clean two-arm contrast. Run `srm_check_multi` on `load_experiment_all()`
    first to confirm the arms you analyse were allocated as designed.
    """
    df = load_experiment_all().filter(
        pl.col("experiment_group").is_in(["control", arm])
    )
    return df.with_columns(
        pl.when(pl.col("experiment_group") == "control")
        .then(pl.lit("control"))
        .otherwise(pl.lit("treatment"))
        .alias("group")
    )


def split_groups(df: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Return (control_df, treatment_df)."""
    control = df.filter(pl.col("group") == "control")
    treatment = df.filter(pl.col("group") == "treatment")
    return control, treatment


def conversion_arrays(df: pl.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Return (control_converted, treatment_converted) as float64 arrays."""
    control, treatment = split_groups(df)
    c = control["converted"].to_numpy().astype(np.float64)
    t = treatment["converted"].to_numpy().astype(np.float64)
    return c, t


def revenue_arrays(df: pl.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Return (control_revenue, treatment_revenue) as float64 arrays."""
    control, treatment = split_groups(df)
    c = control["revenue"].to_numpy().astype(np.float64)
    t = treatment["revenue"].to_numpy().astype(np.float64)
    return c, t


# ════════════════════════════════════════════════════════════════════════
# SANITY CHECKS — SRM
# ════════════════════════════════════════════════════════════════════════


def srm_check(
    n_control: int, n_treatment: int, expected_ratio: float
) -> dict[str, Any]:
    """χ² goodness-of-fit test for Sample Ratio Mismatch on a two-arm pair.

    `expected_ratio` is the DESIGNED share of control within the pair
    (see `designed_control_share`). There is deliberately no 50/50
    default: testing an unequal design against 50/50 always "detects" SRM.

    Returns dict with chi2, p_value, a boolean `srm`, and a verdict.
    SRM indicates randomisation bugs, bot traffic, or pipeline issues —
    if p < 0.01 do NOT trust downstream test results.
    """
    n_total = n_control + n_treatment
    expected = np.array([n_total * expected_ratio, n_total * (1 - expected_ratio)])
    observed = np.array([n_control, n_treatment])
    chi2, p = stats.chisquare(observed, f_exp=expected)
    verdict = (
        "SRM DETECTED — investigate randomisation"
        if p < 0.01
        else "OK — sample split consistent"
    )
    return {"chi2": float(chi2), "p_value": float(p), "srm": bool(p < 0.01), "verdict": verdict}


def srm_check_multi(
    counts: dict[str, int], allocation: dict[str, float] = DESIGNED_ALLOCATION
) -> dict[str, Any]:
    """χ² SRM test across ALL arms against the designed allocation.

    Returns chi2, p_value, srm flag, and a per-arm table (observed share,
    designed share, standardised residual) so you can see WHICH arm is
    mis-allocated, not just that something is wrong.
    """
    arms = list(allocation)
    observed = np.array([counts[a] for a in arms], dtype=np.float64)
    n_total = observed.sum()
    expected = np.array([allocation[a] for a in arms]) * n_total
    chi2, p = stats.chisquare(observed, f_exp=expected)
    per_arm = {
        a: {
            "observed": int(observed[i]),
            "observed_share": float(observed[i] / n_total),
            "designed_share": float(allocation[a]),
            "std_residual": float((observed[i] - expected[i]) / np.sqrt(expected[i])),
        }
        for i, a in enumerate(arms)
    }
    return {
        "chi2": float(chi2),
        "p_value": float(p),
        "srm": bool(p < 0.01),
        "per_arm": per_arm,
    }


# ════════════════════════════════════════════════════════════════════════
# SMALL REUSABLE STATS
# ════════════════════════════════════════════════════════════════════════


def two_proportion_ztest(
    p_control: float, p_treatment: float, n_control: int, n_treatment: int
) -> tuple[float, float]:
    """Pooled two-proportion z-test. Returns (z_stat, two_sided_p_value)."""
    p_pool = (p_control * n_control + p_treatment * n_treatment) / (
        n_control + n_treatment
    )
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_control + 1 / n_treatment))
    z = (p_treatment - p_control) / se if se > 0 else 0.0
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return float(z), float(p_value)


def cohens_h(p1: float, p2: float) -> float:
    """Cohen's h effect size for two proportions."""
    return float(2 * (np.arcsin(np.sqrt(p2)) - np.arcsin(np.sqrt(p1))))


def cohens_d(x1: np.ndarray, x2: np.ndarray) -> float:
    """Pooled Cohen's d effect size for two samples."""
    s_pool = np.sqrt((x1.var(ddof=1) + x2.var(ddof=1)) / 2)
    return float((x2.mean() - x1.mean()) / s_pool) if s_pool > 0 else 0.0


def interpret_magnitude(abs_effect: float) -> str:
    """Cohen convention: <0.2 negligible, <0.5 small, <0.8 medium, else large."""
    if abs_effect < 0.2:
        return "negligible"
    if abs_effect < 0.5:
        return "small"
    if abs_effect < 0.8:
        return "medium"
    return "large"


def print_header(title: str) -> None:
    """Consistent banner for each technique file."""
    print("=" * 70)
    print(f"  {title}")


# ════════════════════════════════════════════════════════════════════════
# PARAMETRIC BOOTSTRAP — resample from a FITTED model, not the data
# ════════════════════════════════════════════════════════════════════════


def parametric_bootstrap_statistic(
    sampler: Callable[[np.random.Generator, int], np.ndarray],
    n: int,
    statistic: Callable[[np.ndarray], float],
    n_boot: int = N_BOOTSTRAP,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """Parametric bootstrap: draw fresh size-n samples from a fitted model.

    ``sampler(rng, n)`` must return n synthetic draws from the fitted
    distribution (e.g. a Normal parameterised by the sample moments). The
    bootstrap distribution of ``statistic`` is then computed entirely under
    the model — powerful when the model is right, misleading when it is not.
    """
    rng = np.random.default_rng(seed)
    out = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        out[i] = statistic(sampler(rng, n))
    return out


# ════════════════════════════════════════════════════════════════════════
# EXPERIMENT TRACKING — kailash-ml ExperimentTracker
# ════════════════════════════════════════════════════════════════════════

TRACKER_STORE_URL = (
    f"sqlite:///{(OUTPUT_DIR / 'experiments.db').resolve().as_posix()}"
)


def track_train_run(
    experiment: str,
    run_name: str,
    params: dict[str, str],
    metrics: dict[str, float],
) -> str:
    """Log one Train-phase run to ExperimentTracker (sync wrapper).

    Returns the run_id. The tracker is closed in a finally block — kailash-ml
    holds the store connection open until close() is called.
    """
    import asyncio

    async def _log() -> str:
        from kailash_ml import ExperimentTracker

        tracker = await ExperimentTracker.create(store_url=TRACKER_STORE_URL)
        try:
            async with tracker.track(
                experiment=experiment, run_name=run_name
            ) as run:
                await run.log_params(params)
                await run.log_metrics(metrics)
                return run.run_id
        finally:
            await tracker.close()

    return asyncio.run(_log())
    print("=" * 70)




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 3.5: Parametric Bootstrap — Resampling from a
#                        Fitted Model
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Distinguish parametric bootstrap (resample from a FITTED
#     distribution) from non-parametric bootstrap (resample the DATA)
#   - See the trade: parametric buys precision when the model is right,
#     and confident error when the model is wrong
#   - Diagnose misspecification: a Normal model on Exponential-shaped
#     waiting times mis-centres the CI for the median by design
#   - Choose a defensible parametric family using the diagnostics from
#     Exercise 2.6 (SD/Mean ≈ 1 → Exponential)
#
# PREREQUISITES: 01_bootstrap_power.py (non-parametric bootstrap, CIs);
#   Exercise 2.6 (Poisson process, Exponential fit)
#
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — what "parametric" buys and what it costs
#   2. Build — inter-arrival gaps from the April 2024 taxi window
#   3. Train — three bootstraps of the median gap: non-parametric,
#      Normal (misspecified), Exponential (defensible)
#   4. Visualise — the three bootstrap distributions side by side
#   5. Apply — a one-morning sample (n = 30): which bootstrap to trust
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go



## THEORY — What "Parametric" Buys and What It Costs


In [ ]:
# The non-parametric bootstrap resamples the DATA with replacement. It
# makes almost no assumptions — but with n = 30 it can only ever recombine
# the 30 values you happened to observe. The tails of the resampled
# statistic are built from the tails of your one sample.
#
# The PARAMETRIC bootstrap resamples from a FITTED MODEL instead:
#   1. Fit a distribution to the sample (e.g. Normal(μ̂, σ̂))
#   2. Draw a fresh synthetic sample of size n from that distribution
#   3. Compute the statistic; repeat
#
# When the model family is RIGHT, the parametric bootstrap is sharper:
# the synthetic samples explore the tails the way the true distribution
# would, not just the way your one sample did. When the model family is
# WRONG, it is confidently wrong — every synthetic sample inherits the
# wrong shape, and the CI centres on the wrong value.
#
# Waiting times make the stakes visible. Exponential gaps are right-skewed:
# the mean exceeds the median by a factor of 1/ln(2) ≈ 1.44. A Normal
# parametric bootstrap of the MEDIAN gap is therefore misspecified BY
# DESIGN — the Normal's median equals its mean, so its bootstrap
# distribution centres ~44% above the median you actually computed.
# Misspecification is not a bigger error bar; it is an error bar around
# the WRONG NUMBER.



## TASK 2 — BUILD: Inter-arrival gaps from the April 2024 window


In [ ]:
print_header("MLFP02 Exercise 3.5: Parametric Bootstrap")

trips = load_taxi_trips()
# TODO: Build the inter-arrival gaps series (seconds between consecutive
# pickups in the April 2024 window)
# Hint: taxi_interarrival_seconds(trips)
gaps = ____

print(f"\n  April 2024 inter-arrival gaps: {len(gaps):,}")
print(f"  Mean gap:   {gaps.mean():.0f}s")
print(f"  Median gap: {np.median(gaps):.0f}s")
print(
    f"  Exponential signature: SD/Mean = {gaps.std(ddof=1) / gaps.mean():.2f} "
    "(Exponential predicts 1.00)"
)
print(
    f"  Skew signature: mean/median = {gaps.mean() / np.median(gaps):.2f} "
    "(Exponential predicts 1/ln2 ≈ 1.44)"
)

# Working sample: n = 60 seeded gaps — small enough that the bootstrap
# method choice visibly matters
rng = np.random.default_rng(RANDOM_SEED)
n_small = 60
# TODO: Draw the seeded working sample of 60 gaps (without replacement)
# Hint: rng.choice(gaps, size=n_small, replace=False)
sample = ____
sample_median = float(np.median(sample))
sample_mean = float(sample.mean())

print(f"\n  Working sample: n = {n_small} (seeded draw)")
print(f"  Sample mean:   {sample_mean:.0f}s")
print(f"  Sample median: {sample_median:.0f}s")
# INTERPRETATION: mean >> median flags right skew. The statistic of
# interest is the MEDIAN wait — the "typical" gap a driver experiences —
# precisely the statistic a symmetric Normal model cannot represent.



### Checkpoint 1


In [ ]:
assert len(sample) == n_small, "Working sample must have n = 60"
assert sample_mean > sample_median, "Expect right skew: mean above median"
assert (sample > 0).all(), "Gaps must be strictly positive"
print("\n>>> Checkpoint 1 passed — sample built\n")



## TASK 3 — TRAIN: Three bootstraps of the median gap


In [ ]:
# (a) Non-parametric: resample the 60 observed gaps
# TODO: Non-parametric bootstrap of the median
# Hint: bootstrap_statistic(sample, lambda x: float(np.median(x)),
#        n_boot=N_BOOTSTRAP, seed=RANDOM_SEED)
boot_nonparam = ____
ci_nonparam = bootstrap_percentile_ci(boot_nonparam)

# (b) Parametric, MISSPECIFIED: Normal(μ̂, σ̂) — symmetric, median = mean
mu_hat = sample_mean
sigma_hat = float(sample.std(ddof=1))
# TODO: Parametric bootstrap under the Normal model
# Hint: parametric_bootstrap_statistic(
#   lambda r, n: r.normal(mu_hat, sigma_hat, size=n),
#   n=n_small, statistic=lambda x: float(np.median(x)),
#   n_boot=N_BOOTSTRAP, seed=RANDOM_SEED)
boot_normal = ____
ci_normal = bootstrap_percentile_ci(boot_normal)

# (c) Parametric, DEFENSIBLE: Exponential — the family validated in 2.6
# TODO: Fit the Exponential model on the sample, then parametric-boot
# the median by drawing from r.exponential(scale=1/lambda_hat, size=n)
# Hint: expo = exponential_mle(sample); scale = 1.0 / expo["lambda"]
expo = ____
boot_exponential = ____
ci_exponential = bootstrap_percentile_ci(boot_exponential)

# Theoretical anchor: median of Exponential(λ) = ln(2)/λ
theoretical_median = float(np.log(2) / expo["lambda"])

print("=== Three bootstraps of the median gap (n=60) ===")
print(f"  Sample median: {sample_median:.0f}s   Sample mean: {sample_mean:.0f}s")
print(f"\n{'Bootstrap':<28} {'Centre':>9} {'95% CI (seconds)':>24} {'Width':>8}")
print("-" * 74)
for label, draws, ci in (
    ("Non-parametric", boot_nonparam, ci_nonparam),
    ("Parametric Normal (!)", boot_normal, ci_normal),
    ("Parametric Exponential", boot_exponential, ci_exponential),
):
    print(
        f"{label:<28} {float(np.median(draws)):>8.0f}s "
        f"[{ci[0]:>7.0f}, {ci[1]:>7.0f}]{ci[1] - ci[0]:>10.0f}"
    )
print(f"\n  Exponential theory: median = ln(2)/λ̂ = {theoretical_median:.0f}s")
# INTERPRETATION: the Normal bootstrap centres on the sample MEAN (its
# own median), ~40% above the sample median — its CI need not bracket
# the statistic you computed. The Exponential bootstrap re-centres near
# ln(2)/λ̂ and overlaps the non-parametric CI. Same loop, same n, same
# seed — the only difference is WHERE new samples come from.



### Log the comparison to ExperimentTracker


In [ ]:
# TODO: Log the three-bootstraps run — sample median/mean, each CI
# centre, and the two CI widths
# Hint: track_train_run(experiment=..., run_name=..., params={...}, metrics={...})
run_id = track_train_run(
    experiment="mlfp02_ex3_05_parametric_bootstrap",
    run_name="median_gap_three_bootstraps",
    params={
        "statistic": "median",
        "n_sample": str(n_small),
        "models": "nonparametric,normal,exponential",
        "n_boot": str(N_BOOTSTRAP),
    },
    metrics={
        "sample_median_s": ____,
        "sample_mean_s": ____,
        "nonparam_ci_centre_s": ____,
        "normal_ci_centre_s": ____,
        "exponential_ci_centre_s": ____,
        "nonparam_ci_width_s": ____,
        "exponential_ci_width_s": ____,
    },
)
print(f"\nLogged training run to ExperimentTracker (run {run_id})")



### Checkpoint 2


In [ ]:
assert len(boot_nonparam) == N_BOOTSTRAP, "Non-parametric draws missing"
assert len(boot_normal) == N_BOOTSTRAP, "Normal parametric draws missing"
assert len(boot_exponential) == N_BOOTSTRAP, "Exponential parametric draws missing"
assert ci_nonparam[0] < sample_median < ci_nonparam[1], (
    "Non-parametric CI must bracket the sample median"
)
assert abs(float(np.median(boot_normal)) - sample_mean) < abs(
    float(np.median(boot_normal)) - sample_median
), "Normal bootstrap must centre near the mean, not the median"
assert ci_exponential[0] < theoretical_median < ci_exponential[1], (
    "Exponential CI must bracket the theoretical median ln(2)/λ̂"
)
print("\n>>> Checkpoint 2 passed — three bootstraps computed\n")



## TASK 4 — VISUALISE: The three bootstrap distributions


In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Histogram(
        x=boot_nonparam,
        nbinsx=60,
        histnorm="probability density",
        name="Non-parametric",
        opacity=0.55,
    )
)
# TODO: Overlay the Normal (misspecified) bootstrap histogram
# Hint: go.Histogram(x=boot_normal, nbinsx=60,
#        histnorm="probability density", name="Parametric Normal
#        (misspecified)", opacity=0.55)
fig.add_trace(____)
# TODO: Overlay the Exponential bootstrap histogram
fig.add_trace(____)
fig.add_vline(
    x=sample_median,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Sample median {sample_median:.0f}s",
)
fig.add_vline(
    x=sample_mean,
    line_dash="dot",
    line_color="red",
    annotation_text=f"Sample mean {sample_mean:.0f}s",
)
fig.update_layout(
    title="Bootstrap Distributions of the Median Gap — Model Choice Moves the Centre",
    xaxis_title="Bootstrapped median gap (seconds)",
    yaxis_title="Density",
    barmode="overlay",
    height=450,
)
fig_path = OUTPUT_DIR / "parametric_bootstrap.html"
fig.write_html(str(fig_path))
print(f"Saved: {fig_path}")



### Checkpoint 3


In [ ]:
assert fig_path.exists(), "Figure must be written"
print("\n>>> Checkpoint 3 passed — visualisation saved\n")



## TASK 5 — APPLY: A One-Morning Sample (n = 30) — Which Bootstrap?


In [ ]:
# A taxi-stand operator (anonymised) instruments a NEW stand and has one
# morning of data: 30 gaps. Management asks for the "typical wait" with
# an interval. At n = 30 the non-parametric bootstrap recombines just 30
# values — its CI is lumpy. The parametric bootstrap can do better IF
# the family is right.
#
# Decision rule used in practice:
#   1. Diagnose the family FIRST (2.6's SD/Mean ≈ 1 check, on as much
#      related data as you can borrow — here the April window).
#   2. Fit the defensible family (Exponential for memoryless gaps).
#   3. Cross-check: if the parametric CI and the non-parametric CI
#      disagree on the CENTRE, the model is wrong, not the data.
#   4. Report both when they agree; distrust the parametric one when
#      they don't.

print("\n--- Application: one-morning sample at a new stand ---")
tiny = gaps[:30]
tiny_median = float(np.median(tiny))
tiny_mean = float(tiny.mean())

# TODO: Non-parametric bootstrap of the median on the 30-gap sample
boot_tiny_np = ____
ci_tiny_np = bootstrap_percentile_ci(boot_tiny_np)

# TODO: Exponential parametric bootstrap on the 30-gap sample
# Hint: expo_tiny = exponential_mle(tiny); then parametric_bootstrap_statistic
expo_tiny = ____
boot_tiny_exp = ____
ci_tiny_exp = bootstrap_percentile_ci(boot_tiny_exp)

print(f"  30 gaps: median {tiny_median:.0f}s, mean {tiny_mean:.0f}s")
print(
    f"  Non-parametric 95% CI: [{ci_tiny_np[0]:.0f}, {ci_tiny_np[1]:.0f}]s  "
    f"width {ci_tiny_np[1] - ci_tiny_np[0]:.0f}s"
)
print(
    f"  Exponential    95% CI: [{ci_tiny_exp[0]:.0f}, {ci_tiny_exp[1]:.0f}]s  "
    f"width {ci_tiny_exp[1] - ci_tiny_exp[0]:.0f}s"
)
width_ratio = (ci_tiny_np[1] - ci_tiny_np[0]) / (ci_tiny_exp[1] - ci_tiny_exp[0])
print(f"  Non-parametric CI is {width_ratio:.2f}× wider")
centres_agree = abs(
    float(np.median(boot_tiny_exp)) - float(np.median(boot_tiny_np))
) < 0.25 * tiny_median
print(f"  Centres agree within 25% of the median? {'YES' if centres_agree else 'NO'}")
print(
    "  At n=30 the parametric CI is SMOOTHER (built from a model, not 30\n"
    "  recombined values), though not narrower here — its value is the\n"
    "  validated shape, not width. It is reportable only because the\n"
    "  family was validated on the full month first: the assumption came\n"
    "  from data, not from convenience."
)



### Checkpoint 4


In [ ]:
assert len(tiny) == 30, "Tiny sample must have n = 30"
assert ci_tiny_np[0] < tiny_median < ci_tiny_np[1], (
    "Non-parametric CI must bracket the tiny-sample median"
)
assert ci_tiny_exp[0] > 0, "Exponential CI must stay positive (durations > 0)"
print("\n>>> Checkpoint 4 passed — application complete\n")



## REFLECTION


✓ Non-parametric bootstrap resamples the DATA; parametric resamples
    from a FITTED MODEL — same loop, different source of new samples
  ✓ Right family: parametric is sharper, especially at small n
  ✓ Wrong family: the Normal bootstrap of a median re-centres on the
    MEAN — an error bar around the wrong number, ~44% high for
    Exponential-shaped waits
  ✓ The Exponential bootstrap of the median reproduces ln(2)/λ̂ and
    agrees with the non-parametric CI
  ✓ Decision rule: diagnose the family first → fit the defensible one →
    cross-check the centre against the non-parametric CI before
    trusting the sharp one

  NEXT: In 06_one_sample_one_tailed.py, you'll formulate directional
  hypotheses — when a one-tailed test is legitimate pre-registration and
  when it is p-hacking.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED (3.5 — Parametric Bootstrap)")
print("=" * 70)
print(
)

print("\n>>> Exercise 3.5 complete — Parametric Bootstrap")

